# NB00 — Baseline WCC Reproduzido (BTCS)
**Budgeted Temporal Case Segmentation for AML Transaction Monitoring**  
Andre da Costa Silva | ITA | 2026

## Objetivo
Reproduzir os números do artigo ICEIS 2026 com erro < 1%.  
Este notebook é a **fundação** de toda a pesquisa — não avance sem validar os resultados aqui.

## Critério de saída
- Coverage, Purity, CR@k, #Cases, edges/case, |E_ind|/|E_k| devem bater com ICEIS (erro < 1%)
- Métricas para k ∈ {1%, 2%, 5%, 10%} em AML100k e AML1M

## Pré-requisito
> Execute em ordem. Runtime recomendado: **T4 GPU** (ou superior para AML1M).

In [ ]:
## %% [BTCS-0] Setup: dependencias, GPU/CPU, Drive e diretorios
import os, sys, re, subprocess
from pathlib import Path

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install(["numpy","pandas","scikit-learn","matplotlib","networkx","psutil","tqdm"])

try:
    import torch
except Exception:
    pip_install(["torch","torchvision","torchaudio",
                 "--index-url","https://download.pytorch.org/whl/cu121"])
    import torch

import numpy as np
import pandas as pd

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| device:", DEVICE)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("[WARN] Drive:", repr(e))

BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path(".").resolve()

# ============================================================
# CAMINHOS CONFIRMADOS NO DRIVE (verificado em marco/2026)
# ============================================================

# --- AML100k ---
# Modelo salvo como "SAGE" (nao "GraphSAGE") com seed 42
AML100K_BASE  = BASE / "DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML100k"
AML100K_ARTIF = AML100K_BASE / "artifacts"
AML100K_PROBS = AML100K_BASE / "results/probs_v4"
AML100K_MODEL = "SAGE"
AML100K_SEED  = 42

# --- AML1M ---
# Modelo salvo como "GraphSAGE" com seed 44
AML1M_BASE    = BASE / "DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M"
AML1M_ARTIF   = AML1M_BASE / "artifacts"
AML1M_PROBS   = AML1M_BASE / "results_aml1m_graphsage_only/probs_v4"
AML1M_MODEL   = "GraphSAGE"
AML1M_SEED    = 44

# --- Saida BTCS ---
BTCS_OUTPUT_DIR = BASE / "GrafosGNN/results"
BTCS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Verificacao ---
print("\n=== Verificacao de arquivos ===")
files_to_check = [
    ("AML100k cache ", AML100K_ARTIF / "edge_data_v4_clean.pt"),
    ("AML100k probs ", AML100K_PROBS / f"{AML100K_MODEL}_seed{AML100K_SEED}_test.npz"),
    ("AML1M   cache ", AML1M_ARTIF   / "edge_data_v4_clean.pt"),
    ("AML1M   probs ", AML1M_PROBS   / f"{AML1M_MODEL}_seed{AML1M_SEED}_test.npz"),
]
all_ok = True
for label, path in files_to_check:
    ok = path.exists()
    all_ok = all_ok and ok
    print(f"  {chr(10003) if ok else chr(10007)}  {label}: {path.name}")
if all_ok:
    print("\n  Todos os arquivos encontrados — pode rodar!")
else:
    print("\n  Algum arquivo nao encontrado — verifique o Drive.")


## 3. Interface Stage 2: `build_cases()` e `evaluate_cases()`

Esta é a **interface central** do benchmark BTCS. Todos os métodos (WCC, B1, B2, B3, BTCS) usam estas funções.

- `build_cases(ranked_edges, full_eval_edges, method, k, params)` → lista de casos
- `evaluate_cases(cases, ground_truth)` → dicionário de métricas

In [ ]:
# ============================================================
# INTERFACE CENTRAL DO BENCHMARK BTCS
# ============================================================

import numpy as np
import time
import psutil
import os
import math


def build_cases(ranked_edges, full_eval_edges, method='wcc', k=0.01, params=None):
    """
    Constrói casos (grupos de transações) a partir do ranking do Stage 1.

    Args:
        ranked_edges: dict com chaves:
            - 'scores': np.array float [E_test] — probabilidades do GraphSAGE
            - 'y':      np.array int   [E_test] — labels verdadeiros
            - 'src':    np.array int   [E_test] — nó origem
            - 'dst':    np.array int   [E_test] — nó destino
        full_eval_edges: dict com chaves 'src', 'dst' (TODAS as arestas do período avaliado)
        method: 'wcc' | 'temporal_wcc' | 'wcc_temporal_ctx' | 'hub_pruned_wcc' | 'btcs'
        k: fração do top-k (ex: 0.01 = 1%)
        params: dict de hiperparâmetros específicos do método

    Returns:
        list de dicts, cada um representando um caso:
            {
              'nodes': set de node_ids,
              'seed_edges': índices das arestas selecionadas no top-k,
              'induced_edges': índices de todas as arestas induzidas pelo caso
            }
    """
    if params is None:
        params = {}

    scores = np.asarray(ranked_edges['scores'], dtype=float)
    src    = np.asarray(ranked_edges['src'],    dtype=np.int64)
    dst    = np.asarray(ranked_edges['dst'],    dtype=np.int64)

    E = len(scores)
    K = max(1, int(math.ceil(k * E)))

    # Selecionar top-K arestas
    order = np.argsort(-scores)
    sel   = order[:K]
    src_sel = src[sel]
    dst_sel = dst[sel]

    if method == 'wcc':
        return _build_wcc(src_sel, dst_sel, sel,
                          full_eval_edges,
                          min_nodes=params.get('min_nodes', 3))
    else:
        raise NotImplementedError(f'Método "{method}" ainda não implementado. '
                                   f'Disponíveis: wcc. '
                                   f'Próximos: temporal_wcc (B1), wcc_temporal_ctx (B2), '
                                   f'hub_pruned_wcc (B3), btcs.')


def _union_find(n):
    """Retorna (parent, rank) para Union-Find com path compression."""
    parent = np.arange(n, dtype=np.int64)
    rank   = np.zeros(n, dtype=np.int64)
    return parent, rank


def _find(parent, a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]  # path compression
        a = parent[a]
    return a


def _union(parent, rank, a, b):
    ra, rb = _find(parent, a), _find(parent, b)
    if ra == rb:
        return
    if rank[ra] < rank[rb]:
        parent[ra] = rb
    elif rank[ra] > rank[rb]:
        parent[rb] = ra
    else:
        parent[rb] = ra
        rank[ra] += 1


def _build_wcc(src_sel, dst_sel, sel_idx, full_eval_edges, min_nodes=3):
    """
    WCC Baseline (B0): Weakly Connected Components sobre o top-k.
    Replica exatamente a lógica do artigo ICEIS 2026.
    """
    # Mapear todos os nós envolvidos no top-k
    nodes_all = np.unique(np.concatenate([src_sel, dst_sel]))
    node_map  = {int(n): i for i, n in enumerate(nodes_all)}
    N = len(nodes_all)

    us = np.fromiter((node_map[int(x)] for x in src_sel), dtype=np.int64)
    vs = np.fromiter((node_map[int(x)] for x in dst_sel), dtype=np.int64)

    parent, rank = _union_find(N)
    for a, b in zip(us, vs):
        _union(parent, rank, int(a), int(b))

    # Componentes
    comp = np.array([_find(parent, i) for i in range(N)], dtype=np.int64)
    _, comp_ids = np.unique(comp, return_inverse=True)
    nodes_per_comp = np.bincount(comp_ids)

    # Filtrar componentes com >= min_nodes nós
    valid_comps = np.where(nodes_per_comp >= min_nodes)[0]

    # gid_of_node: mapeia node_id original -> gid do caso (-1 se não pertence)
    max_node_id = int(nodes_all.max()) + 1
    gid_of_node = -np.ones(max_node_id, dtype=np.int64)
    gid_remap = {int(c): i for i, c in enumerate(valid_comps)}
    for node_id, local_idx in node_map.items():
        c = int(comp_ids[local_idx])
        if c in gid_remap:
            gid_of_node[node_id] = gid_remap[c]

    n_cases = len(valid_comps)
    if n_cases == 0:
        return []

    # Construir casos
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []}
             for _ in range(n_cases)]

    # Arestas semente (top-k que pertencem a casos válidos)
    for i, (u, v) in enumerate(zip(src_sel, dst_sel)):
        g = gid_of_node[int(u)] if int(u) < max_node_id else -1
        if g >= 0:
            cases[g]['seed_edges'].append(int(sel_idx[i]))
            cases[g]['nodes'].add(int(u))
            cases[g]['nodes'].add(int(v))

    # Arestas induzidas (todas as arestas do período dentro dos casos)
    src_full = np.asarray(full_eval_edges['src'], dtype=np.int64)
    dst_full = np.asarray(full_eval_edges['dst'], dtype=np.int64)
    for i, (u, v) in enumerate(zip(src_full, dst_full)):
        gu = gid_of_node[int(u)] if int(u) < max_node_id else -1
        gv = gid_of_node[int(v)] if int(v) < max_node_id else -1
        if gu >= 0 and gu == gv:
            cases[gu]['induced_edges'].append(i)

    return cases


def evaluate_cases(cases, ground_truth, ranked_edges, k):
    """
    Avalia a qualidade dos casos gerados pelo Stage 2.

    Args:
        cases: saída de build_cases()
        ground_truth: dict com:
            - 'y_full': np.array int [E_full] — labels de TODAS as arestas do período
        ranked_edges: mesmo dict passado para build_cases()
        k: fração top-k usada (para calcular CR@k)

    Returns:
        dict com todas as métricas do paper BTCS
    """
    y_full  = np.asarray(ground_truth['y_full'], dtype=int)
    scores  = np.asarray(ranked_edges['scores'], dtype=float)
    y_test  = np.asarray(ranked_edges['y'],      dtype=int)

    E_test  = len(y_test)
    pos_te  = int(y_test.sum())
    K       = max(1, int(math.ceil(k * E_test)))

    if len(cases) == 0:
        return {metric: np.nan for metric in [
            'n_cases', 'coverage', 'purity_induced', 'cr_at_k',
            'edges_per_case_median', 'edges_per_case_p95', 'edges_per_case_max',
            'e_ind_over_ek', 'ocr_b100', 'ocr_b500', 'ocr_b1000'
        ]}

    # ---- Métricas por caso ----
    n_cases = len(cases)
    induced_sizes = [len(c['induced_edges']) for c in cases]
    seed_sizes    = [len(c['seed_edges'])    for c in cases]

    # Positivos induzidos (Coverage)
    pos_induced = sum(int(y_full[c['induced_edges']].sum()) for c in cases)
    coverage    = float(pos_induced / max(pos_te, 1))

    # Pureza induzida (weighted)
    total_induced = sum(induced_sizes)
    purity_induced = float(pos_induced / max(total_induced, 1))

    # Positivos selecionados (para CR@k)
    pos_in_cases_selected = sum(
        int(y_test[c['seed_edges']].sum()) for c in cases
    )
    recall_in_cases = float(pos_in_cases_selected / max(pos_te, 1))
    cr_at_k = float(recall_in_cases / max(coverage, 1e-12)) if coverage > 0 else np.nan

    # E_ind / E_k
    e_ind = sum(induced_sizes)
    e_ind_over_ek = float(e_ind / max(K, 1))

    # edges/case: mediana, p95, max
    ind_arr = np.array(induced_sizes)
    edges_median = float(np.median(ind_arr))
    edges_p95    = float(np.percentile(ind_arr, 95))
    edges_max    = float(ind_arr.max())

    # OCR-B: fração de casos com > B arestas induzidas
    ocr_b100  = float((ind_arr > 100).mean())
    ocr_b500  = float((ind_arr > 500).mean())
    ocr_b1000 = float((ind_arr > 1000).mean())

    return {
        'n_cases':              n_cases,
        'coverage':             coverage,
        'purity_induced':       purity_induced,
        'cr_at_k':              cr_at_k,
        'recall_in_cases':      recall_in_cases,
        'edges_per_case_median': edges_median,
        'edges_per_case_p95':   edges_p95,
        'edges_per_case_max':   edges_max,
        'e_ind_total':          e_ind,
        'e_ind_over_ek':        e_ind_over_ek,
        'ocr_b100':             ocr_b100,
        'ocr_b500':             ocr_b500,
        'ocr_b1000':            ocr_b1000,
    }


print('✓ Interface build_cases() e evaluate_cases() carregadas.')

## 4. Funções Auxiliares de Métricas de Tempo e RAM

In [ ]:
import contextlib

@contextlib.contextmanager
def measure_resources():
    """Context manager para medir tempo e RAM de um bloco de código."""
    proc = psutil.Process(os.getpid())
    mem_before = proc.memory_info().rss / 1024**2  # MB
    t_start = time.perf_counter()
    result = {}
    yield result
    result['time_s']  = time.perf_counter() - t_start
    result['ram_mb']  = proc.memory_info().rss / 1024**2 - mem_before

print('✓ measure_resources() carregado.')

## 5. Carregar Scores do GraphSAGE

Os scores do GraphSAGE foram salvos como `.npz` durante o treinamento:
- `{model}_seed{seed}_test.npz` com campos `p` (probabilidades) e `y` (labels)
- `edge_data_v4_clean.pt` com `ei_all_cpu`, `te_idx`, etc.

In [ ]:
def load_dataset_artifacts(artif_dir, probs_dir, model='GraphSAGE', seed=44, dataset_name='AML100k'):
    """
    Carrega os artefatos salvos do Stage 1 para usar no Stage 2.

    Retorna:
        ranked_edges:    dict com scores, y, src, dst (arestas de teste)
        full_eval_edges: dict com src, dst (todas as arestas do período de teste)
        ground_truth:    dict com y_full (labels de todas as arestas do período)
    """
    artif_dir = Path(artif_dir)
    probs_dir = Path(probs_dir)

    # --- Carregar scores ---
    npz_path = probs_dir / f'{model}_seed{seed}_test.npz'
    if not npz_path.exists():
        raise FileNotFoundError(
            f'Scores não encontrados: {npz_path}\n'
            f'Execute o notebook do Stage 1 (Experimeto_Final_Dissertacao) primeiro.'
        )
    npz  = np.load(npz_path)
    p_te = np.asarray(npz['p'], dtype=float)
    y_te = np.asarray(npz['y'], dtype=int)
    print(f'[{dataset_name}] Scores carregados: {len(p_te)} arestas de teste, '
          f'{y_te.sum()} positivas ({100*y_te.mean():.2f}%)')

    # --- Carregar grafo (edge_index e índices de split) ---
    cache_path = artif_dir / 'edge_data_v4_clean.pt'
    if not cache_path.exists():
        raise FileNotFoundError(
            f'Cache do grafo não encontrado: {cache_path}\n'
            f'Execute a Célula [6] do notebook Stage 1 primeiro.'
        )
    cache = torch.load(cache_path, map_location='cpu')

    # Extrair edge_index e te_idx do cache
    # (adaptado para o formato salvo no Experimeto_Final_Dissertacao)
    ei_all = cache['ei_all_cpu']        # [2, E_all]
    te_idx = cache['te_idx']             # índices das arestas de teste em ei_all
    y_all  = cache.get('y_all_cpu',
             cache.get('y_all', None))   # labels de todas as arestas

    if isinstance(te_idx, torch.Tensor):
        te_idx = te_idx.numpy()
    if isinstance(ei_all, torch.Tensor):
        ei_all = ei_all.numpy()
    if y_all is not None and isinstance(y_all, torch.Tensor):
        y_all = y_all.numpy()

    # Arestas de teste
    src_te = ei_all[0, te_idx]
    dst_te = ei_all[1, te_idx]

    assert len(p_te) == len(src_te), (
        f'Inconsistência: {len(p_te)} scores vs {len(src_te)} arestas de teste'
    )

    ranked_edges = {
        'scores': p_te,
        'y':      y_te,
        'src':    src_te,
        'dst':    dst_te,
    }

    full_eval_edges = {
        'src': src_te,  # Stage 2 usa o mesmo universo de teste
        'dst': dst_te,
    }

    ground_truth = {
        'y_full': y_te,  # labels do período de avaliação
    }

    print(f'[{dataset_name}] Grafo carregado: {ei_all.shape[1]} arestas totais, '
          f'{len(te_idx)} no teste')

    return ranked_edges, full_eval_edges, ground_truth


print('✓ load_dataset_artifacts() carregada.')

In [ ]:
# Carregar artefatos do Stage 1
# Paths e seeds detectados automaticamente na célula de setup
ranked_100k, full_100k, gt_100k = load_dataset_artifacts(
    AML100K_ARTIF, AML100K_PROBS,
    model=AML100K_MODEL, seed=AML100K_SEED,
    dataset_name='AML100k'
)

ranked_1m, full_1m, gt_1m = load_dataset_artifacts(
    AML1M_ARTIF, AML1M_PROBS,
    model=AML1M_MODEL, seed=AML1M_SEED,
    dataset_name='AML1M'
)

## 6. Rodar WCC Baseline para k ∈ {1%, 2%, 5%, 10%}

In [ ]:
KS   = [0.01, 0.02, 0.05, 0.10]
DATASETS = [
    ('AML100k', ranked_100k, full_100k, gt_100k),
    ('AML1M',   ranked_1m,   full_1m,   gt_1m),
]
WCC_PARAMS = {'min_nodes': 3}

rows = []

for ds_name, ranked, full, gt in DATASETS:
    print(f'\n=== {ds_name} ===')
    for k in KS:
        with measure_resources() as res:
            cases = build_cases(
                ranked, full,
                method='wcc',
                k=k,
                params=WCC_PARAMS
            )
            metrics = evaluate_cases(cases, gt, ranked, k)

        row = {
            'dataset': ds_name,
            'method':  'B0_WCC',
            'k%':      f'{int(k*100)}%',
            'k_frac':  k,
            **metrics,
            'time_s':  res['time_s'],
            'ram_mb':  res['ram_mb'],
        }
        rows.append(row)

        print(f'  k={int(k*100)}%: '
              f'#casos={metrics["n_cases"]:,} | '
              f'coverage={metrics["coverage"]:.3f} | '
              f'purity={metrics["purity_induced"]:.4f} | '
              f'CR@k={metrics["cr_at_k"]:.3f} | '
              f'E_ind/E_k={metrics["e_ind_over_ek"]:.2f}x | '
              f'{res["time_s"]:.1f}s')

df_results = pd.DataFrame(rows)
print('\n✓ Baseline WCC concluído!')

## 7. Tabela de Resultados e Validação vs ICEIS

**Critério de saída:** os números abaixo devem bater com o artigo ICEIS com erro < 1%.
Se não baterem, **não avance para os próximos notebooks**.

In [ ]:
# Tabela principal
display_cols = [
    'dataset', 'k%', 'n_cases', 'coverage', 'purity_induced',
    'cr_at_k', 'edges_per_case_median', 'edges_per_case_p95',
    'e_ind_over_ek', 'time_s'
]

df_display = df_results[display_cols].copy()
df_display.columns = [
    'Dataset', 'k%', '#Cases', 'Coverage', 'Purity',
    'CR@k', 'Edges/Case (med)', 'Edges/Case (p95)',
    '|E_ind|/|E_k|', 'Time (s)'
]

print('=== WCC Baseline (B0) — Reprodução ICEIS 2026 ===')
display(df_display.round(4))

# ============================================================
# VALORES DE REFERÊNCIA DO ARTIGO ICEIS — EDITE AQUI
# com os números da Tabela X do paper
# ============================================================
iceis_reference = {
    # ('dataset', 'k%'): {'n_cases': X, 'purity_induced': X, 'e_ind_over_ek': X}
    ('AML1M', '1%'):  {'n_cases': 10546, 'e_ind_over_ek': 1.64},
    ('AML1M', '5%'):  {'n_cases':  4549, 'e_ind_over_ek': 3.82},
    ('AML1M', '10%'): {'n_cases':  1388, 'e_ind_over_ek': 3.60},
}

print('\n=== Verificação vs ICEIS (erro deve ser < 1%) ===')
all_ok = True
for (ds, kpct), ref in iceis_reference.items():
    row = df_results[(df_results['dataset'] == ds) & (df_results['k%'] == kpct)]
    if len(row) == 0:
        print(f'  ⚠️  {ds} k={kpct}: não encontrado nos resultados')
        continue
    row = row.iloc[0]
    for metric, expected in ref.items():
        got = float(row[metric])
        err = abs(got - expected) / max(abs(expected), 1e-9)
        status = '✓' if err < 0.01 else '✗'
        if err >= 0.01:
            all_ok = False
        print(f'  {status} {ds} k={kpct} {metric}: got={got:.3f}, expected={expected:.3f}, err={err*100:.2f}%')

if all_ok:
    print('\n✅ BASELINE VALIDADO — pode avançar para nb01_strong_baselines.ipynb')
else:
    print('\n❌ ATENÇÃO: Algum número não bateu. Verifique os caminhos dos dados e seeds antes de avançar.')

## 8. Métrica OCR-B (Fração de Casos Grandes)

In [ ]:
# OCR-B: fração de casos com > B arestas induzidas
# Este é o problema central que o BTCS resolve!
print('=== OCR-B: Fração de casos acima do limiar B ===')

ocr_cols = ['dataset', 'k%', 'n_cases', 'ocr_b100', 'ocr_b500', 'ocr_b1000',
            'edges_per_case_max']
df_ocr = df_results[ocr_cols].copy()
df_ocr.columns = ['Dataset', 'k%', '#Cases', 'OCR(B=100)', 'OCR(B=500)', 'OCR(B=1000)', 'Max edges/case']
display(df_ocr.round(4))

print('\n→ OCR-B alto (especialmente em k=5% e 10%) é o problema central.')
print('  O método BTCS deve reduzir OCR-B em > 60% (critério mínimo) ou > 85% (critério forte).')

# Plot: distribuição do tamanho dos casos para AML1M, k=5%
print('\n=== Plot: distribuição de edges/caso (AML1M, k=5%) ===')
print('(Rode evaluate_cases com retorno detalhado para visualizar a distribuição)')

## 9. Export e Próximos Passos

In [ ]:
# Salvar resultados
out_csv = (BTCS_OUTPUT_DIR / 'nb00_baseline') / 'b0_wcc_baseline_results.csv'
df_results.to_csv(out_csv, index=False)
print(f'✓ Resultados salvos em: {out_csv}')

# Export LaTeX (para o paper ICDM)
out_tex = (BTCS_OUTPUT_DIR / 'nb00_baseline') / 'b0_wcc_baseline_table.tex'
df_display.round(4).to_latex(out_tex, index=False, escape=False)
print(f'✓ Tabela LaTeX salva em: {out_tex}')

print('\n=== Próximos passos ===')
print('1. ✅ nb00_baseline.ipynb — WCC reproduzido (este notebook)')
print('2. ⏳ nb01_strong_baselines.ipynb — B1 (Temporal-WCC), B2, B3 [Semanas 2-4]')
print('3. ⏳ nb02_btcs_method.ipynb — Grafo Lk + Leiden + Contextualização [Semanas 4-8]')
print('4. ⏳ nb03_ablations.ipynb — 8 ablações sistemáticas [Semanas 8-10]')
print('5. ⏳ nb04_multidataset.ipynb — AMLworld + Libra Bank [Semanas 10-16]')